
# Prepare sc_signal.hdf5 for co-assay

This notebook builds `data/sc_signal.hdf5` with group `atac_100k` for all cells in metadata, matching Higashi's expected format.

It expects per-cell ATAC vectors (binned at the same resolution as `config.JSON`), concatenated across chromosomes in the order `config.chrom_list`.

Supported input patterns:
- Mapping file `data/atac_paths.tsv` with columns: `cellname`, `path` (path to a `.npy` file of length n_bins, or directory containing per-chrom `.npy`).
- Directory `data/atac_binned/` with files `<cellname>.npy` each of length n_bins.
- Directory `data/atac_binned/<cellname>/` with per-chrom files like `chr1.npy`, `chr2.npy`, ...

The cell order is taken from `data/filelist.txt` if available; otherwise from `data/label_info.pickle` `cellname` order.


In [1]:

import os, json, math, h5py, pickle
import numpy as np
import pandas as pd
from pathlib import Path

CONFIG = "config.JSON"
DATA_DIR = "data"
OUT_H5 = os.path.join(DATA_DIR, "sc_signal.hdf5")
SIGNAL_NAME = None  # if None, read from config['coassay_signal'][0]

# Optional inputs
ATAC_MAP = os.path.join(DATA_DIR, "atac_paths.tsv")  # optional mapping file
DIR_FLAT = os.path.join(DATA_DIR, "atac_binned")      # optional directory with <cell>.npy

print("CONFIG:", CONFIG)
print("DATA_DIR:", DATA_DIR)
print("OUT_H5:", OUT_H5)
print("ATAC_MAP:", ATAC_MAP)
print("DIR_FLAT:", DIR_FLAT)


CONFIG: config.JSON
DATA_DIR: data
OUT_H5: data/sc_signal.hdf5
ATAC_MAP: data/atac_paths.tsv
DIR_FLAT: data/atac_binned


In [2]:

# Load config and set parameters
with open(CONFIG, 'r') as f:
    cfg = json.load(f)

chrom_list = cfg["chrom_list"]
res = int(cfg["resolution"])  # base pairs per bin
ref_path = cfg["genome_reference_path"]
coassay = bool(cfg.get("coassay", False))
coassay_signal = cfg.get("coassay_signal", [])
if isinstance(coassay_signal, str):
    coassay_signal = [coassay_signal]
if SIGNAL_NAME is None:
    if not coassay_signal:
        raise ValueError("config['coassay_signal'] must contain the ATAC signal name (e.g., 'atac_100k').")
    SIGNAL_NAME = coassay_signal[0]

print("chrom_list:", chrom_list)
print("resolution:", res)
print("genome_reference_path:", ref_path)
print("signal_name:", SIGNAL_NAME)


chrom_list: ['chr1', 'chr2', 'chr3']
resolution: 100000
genome_reference_path: /share/Data/public/ref_genome/mouse_ref/M23/raw_data/mm10.chr.len
signal_name: atac_100k


In [3]:

# Build per-chrom bin counts and chrom vector of length n_bins
# genome_reference_path expected columns: chrom <tab> size_bp
sizes = pd.read_table(ref_path, sep='	', header=None, names=['chrom','size'])
size_map = {r.chrom: int(r.size) for r in sizes.itertuples(index=False)}

bin_counts = []
for c in chrom_list:
    if c not in size_map:
        raise ValueError(f"Chrom {c} not found in genome_reference_path.")
    nbin = int(math.ceil(size_map[c] / res))
    bin_counts.append(nbin)

chrom_vec = np.concatenate([np.array([c]*n, dtype=object) for c,n in zip(chrom_list, bin_counts)])
print("bin_counts:", dict(zip(chrom_list, bin_counts)))
print("total bins:", chrom_vec.shape[0])


bin_counts: {'chr1': 1955, 'chr2': 1822, 'chr3': 1601}
total bins: 5378


In [4]:

# Determine cell order from filelist.txt if exists; else from label_info.pickle
cells_order = None
filelist = os.path.join(DATA_DIR, 'filelist.txt')
if os.path.exists(filelist):
    paths = [l.strip() for l in open(filelist)]
    def basename_noext(p):
        b = os.path.basename(p)
        # strip common suffixes
        for suf in [".pairs.gz", ".pairs", ".hic", ".cool", ".mcool"]:
            if b.endswith(suf):
                b = b[:-len(suf)]
        return b
    cells_order = [basename_noext(p) for p in paths]
    print(f"Loaded {len(cells_order)} cells from filelist.txt")
else:
    lab = pickle.load(open(os.path.join(DATA_DIR,'label_info.pickle'),'rb'))
    cells_order = [str(x) for x in lab['cellname']]
    print(f"Loaded {len(cells_order)} cells from label_info.pickle")

# Save cell order map for reference (sidecar TSV)
sidecar = os.path.join(DATA_DIR, 'sc_signal_cells.txt')
with open(sidecar, 'w') as f:
    for i,name in enumerate(cells_order):
        f.write(f"{i}\t{name}\n")
print("Wrote cell index mapping:", sidecar)


Loaded 4265 cells from filelist.txt
Wrote cell index mapping: data/sc_signal_cells.txt


In [5]:

# Parameters for ATAC fragment binning
FRAG_DIR = "/zliu_ssd/CHARM/CHARM_brain/data/fragments/atac_frags"
OUT_DIR_ATAC = os.path.join(DATA_DIR, "atac_binned")
OS_MAKEDIRS = True  # create OUT_DIR_ATAC if missing
ONLY_CHR_PREFIX = "chr"  # keep only chroms starting with this prefix

# Prepare chromosome offsets for concatenation order
chrom2offset = {}
offset = 0
for c, n in zip(chrom_list, bin_counts):
    chrom2offset[c] = (offset, n)
    offset += n
TOTAL_BINS = offset
print("TOTAL_BINS:", TOTAL_BINS)

if OS_MAKEDIRS and not os.path.exists(OUT_DIR_ATAC):
    os.makedirs(OUT_DIR_ATAC, exist_ok=True)
print("Fragments dir:", FRAG_DIR)
print("Output binned dir:", OUT_DIR_ATAC)


TOTAL_BINS: 5378
Fragments dir: /zliu_ssd/CHARM/CHARM_brain/data/fragments/atac_frags
Output binned dir: data/atac_binned


In [6]:

# Functions to bin one cell's fragments into a genome-wide vector
import pandas as pd
import numpy as np
import gzip

def frag_path_for_cell(cellname: str) -> str:
    # Files look like R2P2048.atac.frag.bed.gz under FRAG_DIR
    return os.path.join(FRAG_DIR, f"{cellname}.atac.frag.bed.gz")


def bin_cell_from_frag(cellname: str, *, chunksize: int = 2_000_000) -> str:
    in_path = frag_path_for_cell(cellname)
    out_path = os.path.join(OUT_DIR_ATAC, f"{cellname}.npy")

    if not os.path.exists(in_path):
        raise FileNotFoundError(f"Fragment file not found for {cellname}: {in_path}")

    # If already exists and length matches, skip recompute
    if os.path.exists(out_path):
        try:
            arr = np.load(out_path, mmap_mode='r')
            if arr.shape[0] == TOTAL_BINS:
                return out_path
        except Exception:
            pass  # fall through to recompute

    vec = np.zeros(TOTAL_BINS, dtype=np.float32)

    # Read gzipped BED (no header), columns: chrom, start, end, allele, score, strand
    # Use chunked reading to limit memory
    col_names = ["chrom", "start", "end", "allele", "score", "strand"]
    usecols = [0,1,2]

    for chunk in pd.read_table(
        in_path,
        sep='	',
        header=None,
        names=col_names,
        usecols=usecols,
        dtype={0: str, 1: np.int64, 2: np.int64},
        compression='gzip',
        chunksize=chunksize,
        iterator=True,
        comment=None,
        engine='c',
        na_filter=False,
    ):
        # Keep chr* only
        if ONLY_CHR_PREFIX:
            mask_chr = chunk['chrom'].str.startswith(ONLY_CHR_PREFIX)
            chunk = chunk[mask_chr]
        if chunk.empty:
            continue

        # Keep configured chromosomes only
        mask_in = chunk['chrom'].isin(chrom_list)
        if not mask_in.all():
            chunk = chunk[mask_in]
        if chunk.empty:
            continue

        # Midpoint binning
        mids = (chunk['start'].values + chunk['end'].values) // 2
        chroms = chunk['chrom'].values

        # Process per chrom to apply offsets and bounds
        for c in np.unique(chroms):
            idxs = np.where(chroms == c)[0]
            mid_c = mids[idxs]
            off, nbin = chrom2offset[c]
            bins = mid_c // res
            # bounds check
            good = (bins >= 0) & (bins < nbin)
            bins = bins[good]
            # accumulate counts
            if len(bins) > 0:
                np.add.at(vec, off + bins, 1)

    # Save
    np.save(out_path, vec)
    return out_path

# Quick smoke test on first existing cell (if any)
first = None
for name in cells_order:
    if os.path.exists(frag_path_for_cell(name)):
        first = name
        break
print("First fragment found:", first)


First fragment found: R1P10001


In [7]:

# Run binning for all cells in cells_order (multiprocessing)
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

workers = int(cfg.get("cpu_num", mp.cpu_count())) if 'cfg' in globals() else mp.cpu_count()
workers = max(1, min(workers, mp.cpu_count()))
print(f"Using {workers} workers for binning")

ok, missing = 0, []
fut_to_name = {}
with ProcessPoolExecutor(max_workers=workers) as ex:
    for name in cells_order:
        fut = ex.submit(bin_cell_from_frag, name)
        fut_to_name[fut] = name
    for fut in tqdm(as_completed(list(fut_to_name.keys())), total=len(fut_to_name), desc='Binning ATAC to bins'):
        try:
            fut.result()
            ok += 1
        except Exception as e:
            missing.append((fut_to_name[fut], str(e)))

print(f"Binned {ok} cells to {OUT_DIR_ATAC}")


Using 40 workers for binning


Binning ATAC to bins: 100%|██████████| 4265/4265 [00:08<00:00, 480.85it/s]


Binned 4265 cells to data/atac_binned


In [8]:

# Helpers to load per-cell ATAC vectors
import numpy as np

def load_cell_vector_from_map(cellname, chrom_list, bin_counts, mapping_df):
    row = mapping_df.loc[mapping_df['cellname'] == cellname]
    if row.empty:
        raise FileNotFoundError(f"Cell {cellname} not found in mapping file")
    path = row.iloc[0]['path']
    return load_cell_vector_from_path(path, chrom_list, bin_counts)


def load_cell_vector_from_path(path, chrom_list, bin_counts):
    p = Path(path)
    if p.is_file():
        # Expect a flat npy of length sum(bin_counts)
        v = np.load(p)
        if v.ndim != 1:
            raise ValueError(f"Vector at {p} must be 1D, got shape {v.shape}")
        return v.astype(np.float32)
    elif p.is_dir():
        # Expect per-chrom npy files like chr1.npy, chr2.npy ...
        parts = []
        for c, n in zip(chrom_list, bin_counts):
            f = p / f"{c}.npy"
            if not f.exists():
                raise FileNotFoundError(f"Missing per-chrom file: {f}")
            arr = np.load(f)
            if arr.shape[0] != n:
                raise ValueError(f"Chrom {c} length mismatch: expected {n}, got {arr.shape[0]}")
            parts.append(arr.astype(np.float32))
        return np.concatenate(parts, axis=0)
    else:
        raise FileNotFoundError(f"Path does not exist: {p}")


def find_default_cell_path(cellname):
    # Default lookup under DIR_FLAT: either <cell>.npy or directory <cell>/ with per-chrom files
    p1 = Path(DIR_FLAT) / f"{cellname}.npy"
    p2 = Path(DIR_FLAT) / cellname
    if p1.exists():
        return str(p1)
    if p2.exists():
        return str(p2)
    return None


In [9]:

# Prepare loader based on what inputs are available
mapping_df = None
if os.path.exists(ATAC_MAP):
    mapping_df = pd.read_csv(ATAC_MAP, sep='	')
    assert {'cellname','path'}.issubset(mapping_df.columns), "atac_paths.tsv must have columns: cellname, path"
    print(f"Using mapping file with {len(mapping_df)} entries: {ATAC_MAP}")
else:
    print("Mapping file not found; will look under:", DIR_FLAT)


Mapping file not found; will look under: data/atac_binned


In [10]:

# Validate and collect vectors lazily during HDF5 write
from tqdm import tqdm
import h5py

n_bins = int(np.sum(bin_counts))
str_dt = h5py.string_dtype(encoding='utf-8')

# Create or overwrite HDF5
if os.path.exists(OUT_H5):
    os.remove(OUT_H5)

written_cells = []
with h5py.File(OUT_H5, 'w') as f:
    g = f.create_group(SIGNAL_NAME)
    g_bin = g.create_group('bin')
    g_bin.create_dataset('chrom', data=chrom_vec.astype(object), dtype=str_dt)

    missing = []
    widx = 0
    for cell in tqdm(cells_order, desc='Writing cells'):
        try:
            if mapping_df is not None:
                vec = load_cell_vector_from_map(cell, chrom_list, bin_counts, mapping_df)
            else:
                p = find_default_cell_path(cell)
                if p is None:
                    raise FileNotFoundError(f"No default path found for {cell} under {DIR_FLAT}")
                vec = load_cell_vector_from_path(p, chrom_list, bin_counts)
            if vec.shape[0] != n_bins:
                raise ValueError(f"Cell {cell}: expected length {n_bins}, got {vec.shape[0]}")
            g.create_dataset(str(widx), data=vec.astype(np.float32), compression='gzip')
            written_cells.append(cell)
            widx += 1
        except Exception as e:
            missing.append((cell, str(e)))
    if missing:
        print(f"WARNING: {len(missing)} cells failed to write. First 5:\n", "\n".join(map(str, missing[:5])))

# Overwrite sidecar with actually written cells and their indices
with open(os.path.join(DATA_DIR, 'sc_signal_cells.txt'), 'w') as f:
    for i, name in enumerate(written_cells):
        f.write(f"{i}\t{name}\n")
print("Wrote:", OUT_H5, "with", len(written_cells), "cells")


Writing cells: 100%|██████████| 4265/4265 [00:18<00:00, 230.82it/s]

Wrote: data/sc_signal.hdf5 with 4265 cells
